# AffectLab — IEMOCAP private preprocessing

This notebook reads the licensed IEMOCAP archive from the private AffectLab Cloud Storage bucket, extracts annotation/transcript metadata only, creates five speaker-independent folds, and writes processed Hugging Face datasets back to the private bucket. It does not use Google Drive and does not publish raw or row-level data. A GPU runtime is not needed.

In [ ]:
import subprocess

from google.colab import auth

PROJECT_ID = 'cat-behaviour-research'
BUCKET = 'affectlab-research-raluca-biras'
RAW_OBJECT = f'gs://{BUCKET}/data/raw/iemocap/IEMOCAP_full_release.tar.gz'
PROCESSED_OBJECT = f'gs://{BUCKET}/data/processed/iemocap-text-v2-context3'
auth.authenticate_user()
subprocess.run(['gcloud', 'config', 'set', 'project', PROJECT_ID], check=True)
subprocess.run(['gcloud', 'storage', 'objects', 'describe', RAW_OBJECT], check=True)

In [ ]:
import shutil
from pathlib import Path

ARCHIVE = Path('/content/IEMOCAP_full_release.tar.gz')
required_bytes = 20_000_000_000
free_bytes = shutil.disk_usage('/content').free
assert free_bytes >= required_bytes, f'Need at least 20 GB free; found {free_bytes / 1e9:.1f} GB'
if not ARCHIVE.exists():
    subprocess.run(['gcloud', 'storage', 'cp', RAW_OBJECT, str(ARCHIVE)], check=True)
assert ARCHIVE.stat().st_size == 17_695_884_032, 'Archive size does not match the verified source'

In [ ]:
import hashlib

expected_sha256 = 'B4A1EBD19655E54B5DE3F4FF60757EA0F9C0C8C76D50F6B2FEBDBF79F2BC69B1'
digest = hashlib.sha256()
with ARCHIVE.open('rb') as handle:
    while chunk := handle.read(16 * 1024 * 1024):
        digest.update(chunk)
actual_sha256 = digest.hexdigest().upper()
assert actual_sha256 == expected_sha256, f'Archive checksum mismatch: {actual_sha256}'
print('Archive verified:', actual_sha256)

In [ ]:
import base64
import os
import sys

from google.colab import userdata

REPO_URL = 'https://github.com/ralucabiras/emotion-aware-role-play-model.git'
REPO_DIR = Path('/content/emotion-aware-role-play-model')
github_token = userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Add a read-only GITHUB_TOKEN in Colab Secrets and enable notebook access.')
basic_auth = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
auth_option = f'http.extraHeader=Authorization: Basic {basic_auth}'
if not REPO_DIR.exists():
    subprocess.run(['git', '-c', auth_option, 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), '-c', auth_option, 'pull', '--ff-only'], check=True)
del github_token, basic_auth, auth_option
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-ml.txt')], check=True)

In [ ]:
OUTPUT_DIR = Path('/content/iemocap-text-v2-context3')
subprocess.run(
    [
        sys.executable, '-m', 'ml.preprocessing.iemocap',
        '--archive', str(ARCHIVE),
        '--output-dir', str(OUTPUT_DIR),
        '--archive-sha256', expected_sha256,
        '--archive-generation', '1785934282898479',
    ],
    check=True,
)

In [ ]:
import json

manifest = json.loads((OUTPUT_DIR / 'manifest.json').read_text())
print('Source labels:', manifest['source_label_counts'])
for task_name, task in manifest['tasks'].items():
    print(f'\n{task_name}: {task["labels"]}')
    display(task['folds']['5'])

In [ ]:
subprocess.run(
    ['gcloud', 'storage', 'rsync', '--recursive', str(OUTPUT_DIR), PROCESSED_OBJECT],
    check=True,
)
subprocess.run(['gcloud', 'storage', 'ls', f'{PROCESSED_OBJECT}/manifest.json'], check=True)
print('Private processed dataset uploaded:', PROCESSED_OBJECT)

## Completion checks

- Confirm both `benchmark_4` and `affectlab_6` appear.
- In every fold, train, validation, and test speaker lists must be disjoint.
- Fold 5 uses Sessions 1–3 for training, Session 4 for validation, and Session 5 for testing.
- Keep the Cloud Storage bucket private. Do not upload the archive, processed rows, or checkpoints to GitHub or Hugging Face.
- Delete `/content/IEMOCAP_full_release.tar.gz` when the Colab runtime is no longer needed; Colab normally discards it with the runtime.